# Goldset Final Grade 결정

3번의 LLM Judge 결과를 합산하여 최종 grade를 확정하고,
query_id별 grade 2~3 문서 충분성을 점검한다.

## 판정 규칙
| 조건 | 결과 | confidence |
|---|---|---|
| 완전 일치 (3/3 동일) | gold 포함 | HIGH |
| 다수결 (2/3 동일, outlier 차이 무관) | gold 포함 | MEDIUM |
| 완전 불일치 (3개 모두 다른 값) | gold 제외 | LOW |

## 데이터 소스
| Judge | 파일 |
|---|---|
| 1st | `goldset_all_scenarios_recovered.json` |
| 2nd | `goldset_all_scenarios_reeval.json` (S01~S10) + `goldset_all_scenarios_rejudge.json` (S11~S21) |
| 3rd | `goldset_all_scenarios_reeval2.json` |

## 1. 데이터 로드

In [49]:
import json
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

DATA_DIR = Path("../dataset")

def load_json(path: Path) -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

judge1 = load_json(DATA_DIR / "goldset_s01_s02_s03_s08_s15_s17_reeval1.json")

# 2nd judge: S01~S10은 reeval, S11~S21은 rejudge
# S01_S10 = {f"S{i:02d}" for i in range(1, 11)}
judge2  = load_json(DATA_DIR / "goldset_s01_s02_s03_s08_s15_s17_reeval2.json")
# rejudge = load_json(DATA_DIR / "goldset_all_scenarios_reeval2.json")
# judge2  = [r for r in reeval if r["query_id"] in S01_S10] + rejudge

judge3 = load_json(DATA_DIR / "goldset_s01_s02_s03_s08_s15_s17_reeval3.json")

print(f"1st judge: {len(judge1):,}건")
print(f"2nd judge: {len(judge2):,}건" ) # (reeval {len([r for r in reeval if r['query_id'] in S01_S10])} + rejudge {len(rejudge)})")
print(f"3rd judge: {len(judge3):,}건")

1st judge: 477건
2nd judge: 477건
3rd judge: 477건


In [50]:
# 각 Judge 파일의 Failed(relevance_grade=None) 건수 확인
from collections import defaultdict

def failed_summary(records: list, label: str):
    total  = len(records)
    failed = [r for r in records if r.get("relevance_grade") is None]
    by_qid = defaultdict(int)
    for r in failed:
        by_qid[r["query_id"]] += 1

    print(f"=== {label}  (총 {total:,}건 / 실패 {len(failed):,}건) ===")
    if failed:
        print(f"  {'query_id':>10}  {'failed':>7}")
        print(f"  {'-'*20}")
        for qid in sorted(by_qid):
            print(f"  {qid:>10}  {by_qid[qid]:>7}")
    else:
        print("  실패 없음")
    print()

failed_summary(judge1, "1st judge  (goldset_all_scenarios.json)")
failed_summary(judge2, "2nd judge  (reeval S01~S10 + rejudge S11~S21)")
failed_summary(judge3, "3rd judge  (goldset_all_scenarios_reeval2.json)")

=== 1st judge  (goldset_all_scenarios.json)  (총 477건 / 실패 0건) ===
  실패 없음

=== 2nd judge  (reeval S01~S10 + rejudge S11~S21)  (총 477건 / 실패 0건) ===
  실패 없음

=== 3rd judge  (goldset_all_scenarios_reeval2.json)  (총 477건 / 실패 0건) ===
  실패 없음



## 2. 3개 Judge 결과 병합

In [51]:
# isbn + query_id → grade 매핑 (None 이외는 모두 int로 정규화)
def to_int_grade(v):
    return int(v) if v is not None else None

g1 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge1}
g2 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge2}
g3 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge3}

all_keys = set(g1) | set(g2) | set(g3)
missing  = {k for k in all_keys if not (k in g1 and k in g2 and k in g3)}
print(f"전체 문서 수: {len(all_keys):,}")
print(f"3개 judge 모두 있는 문서: {len(all_keys) - len(missing):,}")
if missing:
    print(f"⚠️  일부 judge 누락: {len(missing)}건 — 해당 문서는 제외됩니다")

전체 문서 수: 477
3개 judge 모두 있는 문서: 477


## 3. 최종 Grade 결정 함수

In [52]:
def decide_final_grade(grades: List[Optional[int]]) -> Dict[str, Any]:
    # None(LLM 오류) 포함 시 판정 불가 → 제외
    if any(g is None for g in grades):
        return {"final_grade": None, "confidence": "LOW", "gold_status": "excluded"}

    counter = Counter(grades)
    majority_grade, majority_count = counter.most_common(1)[0]

    # 완전 일치 (3/3) → HIGH
    if majority_count == 3:
        return {"final_grade": majority_grade, "confidence": "HIGH",   "gold_status": "included"}

    # 다수결 (2/3 동일 값) → MEDIUM  ※ outlier와의 차이는 무관
    if majority_count == 2:
        return {"final_grade": majority_grade, "confidence": "MEDIUM", "gold_status": "included"}

    # 완전 불일치 (3개 모두 다른 값) → 제외
    return {"final_grade": None, "confidence": "LOW", "gold_status": "excluded"}

## 4. 전체 문서에 판정 적용

In [53]:
# 메타데이터는 1st judge 기준으로 사용
meta = {(r["isbn"], r["query_id"]): r for r in judge1}

records = []
for key in sorted(all_keys - missing):
    isbn, query_id = key
    grades = [g1[key], g2[key], g3[key]]
    verdict = decide_final_grade(grades)

    base = meta[key]
    records.append({
        "query_id"   : query_id,
        "isbn"       : isbn,
        "title"      : base.get("title", ""),
        "grade_j1"   : grades[0],
        "grade_j2"   : grades[1],
        "grade_j3"   : grades[2],
        **verdict,
    })

df = pd.DataFrame(records)
print(df["gold_status"].value_counts().to_string())
print(f"\n포함: {(df['gold_status']=='included').sum():,}  /  제외: {(df['gold_status']=='excluded').sum():,}")

gold_status
included    476
excluded      1

포함: 476  /  제외: 1


## 5. 판정 결과 분포

In [54]:
print("=== confidence 분포 ===")
print(df["confidence"].value_counts().to_string())

print("\n=== final_grade 분포 (포함된 문서) ===")
included = df[df["gold_status"] == "included"]
print(included["final_grade"].value_counts().sort_index().to_string())

print("\n=== query_id별 포함/제외 ===")
summary = df.groupby("query_id")["gold_status"].value_counts().unstack(fill_value=0)
summary["total"] = summary.sum(axis=1)
print(summary.to_string())

# ── query_id별 grade 분포 상세 테이블 ──────────────────────────────────
# "excluded" = 3개 judge 합의 실패로 goldset에서 제외된 문서
#              (LLM API 실패와 무관 — judge 파일의 Failed 건수와 다름)
print("\n[query_id별 relevance_grade 분포]")

df["_label"] = df.apply(
    lambda r: f"grade {int(r['final_grade'])}" if r["gold_status"] == "included" else "excluded",
    axis=1,
)
pivot = df.groupby(["query_id", "_label"]).size().unstack(fill_value=0)
df.drop(columns=["_label"], inplace=True)

grade_cols = [f"grade {i}" for i in range(4) if f"grade {i}" in pivot.columns]
col_order  = grade_cols + (["excluded"] if "excluded" in pivot.columns else [])
pivot      = pivot.reindex(columns=col_order, fill_value=0)
pivot["total"] = pivot.sum(axis=1)

col_w = 9
qid_w = 10
header = f"  {'query_id':>{qid_w}}" + "".join(f"  {c:>{col_w}}" for c in col_order) + f"  {'total':>{col_w}}"
sep    = "-" * (len(header))
print(header)
print(sep)
for qid, row in pivot.iterrows():
    vals = "".join(f"  {int(row[c]):>{col_w}}" for c in col_order)
    print(f"  {qid:>{qid_w}}{vals}  {int(row['total']):>{col_w}}")

=== confidence 분포 ===
confidence
HIGH      335
MEDIUM    141
LOW         1

=== final_grade 분포 (포함된 문서) ===
final_grade
0.0     72
1.0    347
2.0     40
3.0     17

=== query_id별 포함/제외 ===
gold_status  excluded  included  total
query_id                              
S01                 0        75     75
S02                 0        83     83
S03                 0        70     70
S08                 0        86     86
S15                 0        80     80
S17                 1        82     83

[query_id별 relevance_grade 분포]
    query_id    grade 0    grade 1    grade 2    grade 3   excluded      total
------------------------------------------------------------------------------
         S01          6         63          5          1          0         75
         S02         12         68          3          0          0         83
         S03         19         49          2          0          0         70
         S08         10         67          8          1          0     

## 6. query_id별 grade 2~3 문서 충분성 점검

**최소 기준**: query_id별 grade ≥ 2 문서 3개 이상

In [55]:
MIN_GRADE  = 2
MIN_COUNT  = 3

high_grade = included[included["final_grade"] >= MIN_GRADE]
grade_count = high_grade.groupby("query_id").size().rename(f"grade>={MIN_GRADE}_count")

# 전체 query_id 기준으로 merge (없으면 0)
all_qids = pd.Series(sorted(df["query_id"].unique()), name="query_id")
check = all_qids.to_frame().merge(grade_count.reset_index(), on="query_id", how="left").fillna(0)
check[f"grade>={MIN_GRADE}_count"] = check[f"grade>={MIN_GRADE}_count"].astype(int)
check["sufficient"] = check[f"grade>={MIN_GRADE}_count"] >= MIN_COUNT

print(f"=== grade ≥ {MIN_GRADE} 문서 수 (최소 {MIN_COUNT}개 기준) ===")
print(check.to_string(index=False))

insufficient = check[~check["sufficient"]]
if insufficient.empty:
    print(f"\n✅ 모든 query_id가 최소 기준({MIN_COUNT}개) 충족")
else:
    print(f"\n⚠️  기준 미달 query_id ({len(insufficient)}개) → 재검토 필요:")
    print(insufficient.to_string(index=False))

=== grade ≥ 2 문서 수 (최소 3개 기준) ===
query_id  grade>=2_count  sufficient
     S01               6        True
     S02               3        True
     S03               2       False
     S08               9        True
     S15               4        True
     S17              33        True

⚠️  기준 미달 query_id (1개) → 재검토 필요:
query_id  grade>=2_count  sufficient
     S03               2       False


## 7. 제외된 문서 상세 (LOW 판정)

In [56]:
excluded = df[df["gold_status"] == "excluded"].copy()
excluded["grades"] = excluded.apply(lambda r: [r.grade_j1, r.grade_j2, r.grade_j3], axis=1)

print(f"제외 문서 총 {len(excluded)}건")
print("\n=== query_id별 제외 수 ===")
print(excluded["query_id"].value_counts().sort_index().to_string())

print("\n=== 제외 문서 샘플 (상위 10개) ===")
pd.set_option("display.max_colwidth", 25)
excluded[["query_id", "isbn", "title", "grades"]]

제외 문서 총 1건

=== query_id별 제외 수 ===
query_id
S17    1

=== 제외 문서 샘플 (상위 10개) ===


,query_id,isbn,title,grades
262,S17,9788990828699,"굿바이, 굿 보이 - 상스 청소년 소설","[1, 3, 2]"


## 8. 최종 Goldset 저장

In [57]:
# 포함된 문서에 원본 메타데이터 결합
included_keys = set(zip(included["isbn"], included["query_id"]))

final_records = []
for key in sorted(included_keys):
    isbn, query_id = key
    base  = meta[key].copy()
    row   = df[(df["isbn"] == isbn) & (df["query_id"] == query_id)].iloc[0]

    base["final_grade"] = int(row["final_grade"])
    base["confidence"]  = row["confidence"]
    base["grade_j1"]    = int(row["grade_j1"])
    base["grade_j2"]    = int(row["grade_j2"])
    base["grade_j3"]    = int(row["grade_j3"])
    final_records.append(base)

out_path = DATA_DIR / "goldset_final_6.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_records, f, ensure_ascii=False, indent=2)

# out_jsonl = DATA_DIR / "goldset_final.jsonl"
# with open(out_jsonl, "w", encoding="utf-8") as f:
#     for r in final_records:
#         f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"저장 완료: {out_path}  ({len(final_records):,}건)")
# print(f"저장 완료: {out_jsonl}")

저장 완료: ../dataset/goldset_final_6.json  (476건)


## 9. 최종 Goldset 검증

In [58]:
final_df = pd.DataFrame(final_records)

print("=== 최종 Goldset 요약 ===")
print(f"총 문서 수    : {len(final_df):,}")
print(f"query_id 수   : {final_df['query_id'].nunique()}")

print("\n=== query_id별 grade 분포 ===")
pivot = final_df.groupby(["query_id", "final_grade"]).size().unstack(fill_value=0)
pivot.columns = [f"grade_{c}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
if "grade_2" in pivot.columns and "grade_3" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_2"] + pivot["grade_3"]
elif "grade_2" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_2"]
elif "grade_3" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_3"]
print(pivot.to_string())

print("\n=== confidence 분포 ===")
print(final_df["confidence"].value_counts().to_string())

=== 최종 Goldset 요약 ===
총 문서 수    : 476
query_id 수   : 6

=== query_id별 grade 분포 ===
          grade_0  grade_1  grade_2  grade_3  total  grade_2+3
query_id                                                      
S01             6       63        5        1     75          6
S02            12       68        3        0     83          3
S03            19       49        2        0     70          2
S08            10       67        8        1     86          9
S15            16       60        4        0     80          4
S17             9       40       18       15     82         33

=== confidence 분포 ===
confidence
HIGH      335
MEDIUM    141
